# GX_10
# Lee Holograms

In this exercise, you are going to experiment with building holograms from amplitude objects based on Lee sampling. You are provided with a phase object that you wish to create and propagate to retrieve an original image, and will compare three different ways of accomplishing this.

In [ ]:
# - No modification necessary -

import numpy as np
import matplotlib.pyplot as plt

from scipy.fft import fft2, ifft2, fftshift, ifftshift, fftfreq
from skimage import data, transform

In [ ]:
# - No modification necessary -

# ============================================================
# Simulation Parameters
# ============================================================

wavelength = 532e-9          # 532 nm
k = 2 * np.pi / wavelength

N = 512                      # simulation size
dx = 8e-6                    # pixel pitch

z_object = -0.02             # backpropagation distance
z_propagate = -z_object           # forward propagation distance

padding = 128

carrier_frequency = 1 / (8 * dx)

# ============================================================
# Spatial Coordinates
# ============================================================

x = (np.arange(N) - N // 2) * dx
y = (np.arange(N) - N // 2) * dx

X, Y = np.meshgrid(x, y)

# ============================================================
# Frequency Coordinates
# ============================================================

fx = fftshift(fftfreq(N, d=dx))
fy = fftshift(fftfreq(N, d=dx))

FX, FY = np.meshgrid(fx, fy)

# Angular frequencies
KX = 2 * np.pi * FX
KY = 2 * np.pi * FY

# ============================================================
# Plotting Utilities
# ============================================================

def show_intensity(field, title="Intensity", cmap="inferno"):
    intensity = np.abs(field) ** 2

    plt.figure(figsize=(6, 6))
    plt.imshow(intensity, cmap=cmap)
    plt.title(title)
    plt.colorbar()
    plt.axis("off")
    plt.show()


def show_phase(field, title="Phase", cmap="twilight"):
    phase = np.angle(field)

    plt.figure(figsize=(6, 6))
    plt.imshow(phase, cmap=cmap)
    plt.title(title)
    plt.colorbar()
    plt.axis("off")
    plt.show()


def show_amplitude(field, title="Amplitude", cmap="viridis"):
    amplitude = np.abs(field)

    plt.figure(figsize=(6, 6))
    plt.imshow(amplitude, cmap=cmap)
    plt.title(title)
    plt.colorbar()
    plt.axis("off")
    plt.show()
    
def show_field(field, title="Field"):

    fig, ax = plt.subplots(1, 2, figsize=(10, 4))

    amp = ax[0].imshow(np.abs(field), cmap="viridis")
    ax[0].set_title(f"{title} Amplitude")
    ax[0].axis("off")
    plt.colorbar(amp, ax=ax[0])

    phase = ax[1].imshow(
        np.angle(field),
        cmap="twilight",
        vmin=-np.pi,
        vmax=np.pi
    )

    ax[1].set_title(f"{title} Phase")
    ax[1].axis("off")
    plt.colorbar(phase, ax=ax[1])

    plt.tight_layout()
    plt.show()

def show_spectrum(field, title="Fourier Magnitude"):
    spectrum = np.log10(np.abs(fftshift(fft2(field))) + 1e-6)

    plt.figure(figsize=(6, 6))
    plt.imshow(spectrum, cmap="magma")
    plt.title(title)
    plt.colorbar()
    plt.axis("off")
    plt.show()
    
# ============================================================
# Angular Spectrum Propagation
# ============================================================

def angular_spectrum_propagation(field, z):
    """
    Propagate a complex field using the angular spectrum method.
    """

    kz = np.sqrt(
        np.maximum(
            0,
            k**2 - KX**2 - KY**2
        )
    )

    H = np.exp(1j * kz * z)

    field_spectrum = fftshift(fft2(field))

    propagated_spectrum = field_spectrum * H

    propagated_field = ifft2(ifftshift(propagated_spectrum))

    return propagated_field

In [ ]:
# - No modification necessary -

image = data.camera()
image = transform.resize(image, (N - 2 * padding, N - 2 * padding))
image = image.astype(np.float64)

image -= image.min()
image /= image.max()

phase_image = 2 * np.pi * image

padded_phase = np.zeros((N, N))

start = padding
end = N - padding
padded_phase[start:end, start:end] = phase_image

object_field = np.exp(1j * padded_phase)
show_field(object_field, "Initial Object")

backpropagated_field = angular_spectrum_propagation(object_field, z_object)
show_field(backpropagated_field, "Backpropagated Object")

input_phase = np.angle(backpropagated_field)
input_field = np.exp(1j * input_phase)
show_field(input_field, "Final Object")

# Part 1
## Direct Propagation

The first and most simple method to create your phase object and propagate it would be to use a device such as a Liquid Crystal on Silicon (LCoS) Spatial Light Modulator (SLM). This device allows you to directly encode phase into a field. In this section the code required to propagate the phase object to retrieve the output is already provided, so you will not need to do anything. This section is merely provided as a reference for what is to come.

In [ ]:
# - No modification necessary -

phase_output = angular_spectrum_propagation(
    input_field,
    z_propagate
)

show_field(phase_output, "Direct Phase Output")

# Part 2
## Continuous Lee Hologram

If you are not able to source a phase SLM for your application, another option is to use a spatial amplitude modulation device (think, for example, of a classic liquid crystal display). If you want to use an amplitude modulation device to encode a phase object, however, you must use a trick to convert between the two effects.

In this workbook you will be looking at using Lee holograms, which create their phase modulation by overlaying a sinusoidal frequency grating on the amplitude object. The field is then filtered in Fourier space to convert the amplitude envelope of the object into phase, and then the field is returned to real space as a phase object. The origin of this effect can be visually seen in the following example equation for amplitude:

$$
\text{Amplitude}: f(x,y) = \frac{1}{2} [ 1 + \cos{2\pi(x-y)\nu_0 + \phi (x,y)} ] = \frac{1}{2} + \frac{1}{4} e^{i 2 \pi (y-x) \nu_0} e^{-i\phi(x,y)} + \frac{1}{4} e^{i 2 \pi (x-y) \nu_0} e^{i\phi(x,y)}
$$

To build a Lee hologram yourself, you will first start by implementing the function generate_continuous_lee_hologram. This function will take the phase field $\phi$ as an argument and should return the encoded amplitude image. Note that for this you should follow the provided equation, but you can disregard y as we will do a single-axis tilt for this example. After you have implemented the function you can visualize your hologram with the provided script.

Once you have your amplitude object, you must pass it through a 4F system to filter out the first order. Since we are doing this all computationally, we will do this by first taking a fourier transform, then isolating and shifting the first order back to the center, and finally doing an inverse fourier transform. You should create the required filter in the function make_fourier_filter based on the translation of the first order and the desired filter size, and implement the full reconstruction logic in the function reconstruct_from_hologram. After you have done this you can run the remaining visualization code to see what your outputs look like.

In [ ]:
def generate_continuous_lee_hologram(field):
    """
    Generate a continuous Lee hologram using cosine carrier encoding.
    """

    raise NotImplementedError

In [ ]:
# - No modification necessary -

continuous_hologram = generate_continuous_lee_hologram(
    input_field
)

show_amplitude(continuous_hologram, "Continuous Lee Hologram Amplitude")
show_spectrum(continuous_hologram, "Continuous Lee Hologram Spectrum")

In [ ]:
def make_fourier_filter(
    center_fx,
    center_fy,
    radius
):
    """
    Circular Fourier mask.
    """

    raise NotImplementedError

In [ ]:
# - No modification necessary -

filter_radius = 0.8 * carrier_frequency
first_order_filter = make_fourier_filter(carrier_frequency, 0, filter_radius)

show_amplitude(first_order_filter, "First Order Filter")


In [ ]:
def reconstruct_from_hologram(
    hologram,
    fourier_filter,
    shift_pixels_x,
    shift_pixels_y=0
):
    """
    Simulate a 4F optical system.

    Steps:
    1. Fourier transform hologram
    2. Isolate first diffraction order
    3. Shift isolated order back to center
    4. Inverse transform to recover field
    """

    raise NotImplementedError

In [ ]:
# - No modification necessary -

shift_pixels_x = int(np.round(carrier_frequency / (fx[1] - fx[0])))

continuous_reconstruction = reconstruct_from_hologram(
    continuous_hologram,
    first_order_filter,
    shift_pixels_x
)

show_field(continuous_reconstruction, "Continuous Lee Hologram Reconstruction")

In [ ]:
# - No modification necessary -

continuous_output = angular_spectrum_propagation(
    continuous_reconstruction,
    z_propagate
)

show_field(continuous_output, "Continuous Lee Hologram Output")

# Part 3
## Binary Lee Hologram

You have now seen that from a continuous amplitude object you can reconstruct a phase object. But what if you only have a binary amplitude modulation system? Most commonly, this scheme is used in Digital Micromirror Devices (DMDs). These devices are often highly advantageous relative to liquid crystal technologies due to their significantly higher modulation speeds, but each pixel can only be either on or off. 

To generate a Lee hologram with a device like this, we use thresholding to binarize the continuous hologram that we desire to show. Implement that behavior below in the function binarize_hologram and use the provided visualization scripts to test if the reconstruction methodology still works.

In [ ]:
def binarize_hologram(
    hologram,
    threshold=0.5
):
    raise NotImplementedError

In [ ]:
# - No modification necessary -

binary_hologram = binarize_hologram(
    continuous_hologram
)

show_amplitude(binary_hologram, "Binary Lee Hologram Amplitude")
show_spectrum(binary_hologram, "Binary Hologram Spectrum")

In [ ]:
# - No modification necessary -

binary_reconstruction = reconstruct_from_hologram(
    binary_hologram,
    first_order_filter,
    shift_pixels_x
)

show_field(binary_reconstruction, "Binary Lee Hologram Reconstruction")


In [ ]:
# - No modification necessary -

binary_output = angular_spectrum_propagation(
    binary_reconstruction,
    z_propagate
)

show_field(binary_output, "Binary Lee Hologram Output")

# Part 4
## Comparison

Now that you have implemented these three different methods for generating a phase object, you will answer the following questions to compare their outputs and advantages/disadvantages. Use the provided visualization script to see all three results side by side.

1) Why do we seem to lose higher spatial frequencies in the outputs that were reconstructed from an amplitude object?
2) What factors would you control in a physical setup to improve the isolation of the first order?
3) Read sections 2.2 and 3.1 of the following article: https://iopscience.iop.org/article/10.1088/2515-7647/ad8617. In your own words, explain why binarizing a Lee hologram still produces a valid (but worse than in the continuous case) reconstruction of the field.

In [ ]:
# - No modification necessary -

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

outputs = [
    phase_output,
    continuous_output,
    binary_output
]

titles = [
    "Pure Phase",
    "Continuous Lee",
    "Binary Lee"
]

for i in range(len(outputs)):
    intensity = np.abs(outputs[i]) ** 2
    phase = np.angle(outputs[i])

    axes[0, i].imshow(intensity, cmap="inferno")
    axes[0, i].set_title(titles[i] + " Intensity")
    axes[0, i].axis("off")

    axes[1, i].imshow(phase, cmap="twilight")
    axes[1, i].set_title(titles[i] + " Phase")
    axes[1, i].axis("off")

plt.tight_layout()
plt.show()

## Discussion
TODO

# Bonus
In this exercise we have only encoded phase in our binary Lee holograms. Should we also be able to control amplitude simultaneously? If so, how would we do it? (Hint: Look at the provided article for discussion question 3)

## Discussion
TODO